# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

Simple words, honest numbers. Everything below is **observed** on one sampled month (March 2026) and is **directional**, for decision-support only.

**Plan (3 small things):** (1) check two signals with bucket tables, (2) encode ONE rule and write the ranked queue, (3) review my top picks with a skeptic's eye.

## 1. My rule and its reason codes

### The rule in plain words
> **"Pages that rank well enough to be seen but get fewer clicks than pages at the same position usually do, are worth a title/meta rewrite. The more impressions they have, the more clicks a rewrite could win back."**

This is the CTR-vs-position logic behind FlyRank's CTR-fix flag, built by hand with honest thresholds.

- **Score** = estimated missed clicks = `max(expected_ctr_at_this_position - actual_ctr, 0) x impressions`
- **Eligible:** at least 100 impressions in the window and average position between 1 and 20
- **ONE reason code:** `CTR_BELOW_POSITION_NORM`
- **Action label:** `REWRITE_TITLE_AND_META`
- Pages with no gap or not eligible get score 0, code `NO_SIGNAL`, action `NO_ACTION` (this is the absence of the rule, not a second rule).

### Time window (no future inputs)
The score only uses the **first 21 days** of March 2026. The last 10 days are never used for scoring; they are used once in Section 4 as a directional persistence check.

### Two signals the rule leans on (checked before trusting the rule)
1. **CTR falls as position gets worse** (flag-linked: the CTR-fix logic).
2. **Higher search volume means more impressions** (flag-linked: the volume logic behind quick-win).

Each gets a bucket table with n and a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [1]:
import os, json
import numpy as np
import pandas as pd
from huggingface_hub import login

# Token comes from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Add it in Colab -> Secrets and enable notebook access.")

os.environ["HF_TOKEN"] = token
login(token=token)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
!pip install -q huggingface_hub pandas pyarrow

In [3]:
storage_options = {"token": token}
BASE = "hf://datasets/FlyRank/internship-warehouse"

df_raw = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options=storage_options)
df_content = pd.read_parquet(f"{BASE}/dim_content.parquet", storage_options=storage_options)
print("Performance rows:", len(df_raw), "| content rows:", len(df_content))

Performance rows: 9841378 | content rows: 519606


In [4]:
# ---- Prep: check columns, keep GSC-available rows, split into observation window vs later window ----
KEY = ["client_hash_id", "content_hash_id"]
need = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
        "gsc_sum_position", "client_has_gsc", "gsc_data_available"]
missing_cols = [c for c in need if c not in df_raw.columns]
assert not missing_cols, f"Missing columns in fact table: {missing_cols} -> rename them in this cell"

df = df_raw[(df_raw["client_has_gsc"] == True) & (df_raw["gsc_data_available"] == True)].copy()
df["report_date"] = pd.to_datetime(df["report_date"])
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    df[c] = df[c].fillna(0)
df["active"] = (df["gsc_impressions"] > 0).astype(int)

START = df["report_date"].min()
CUTOFF = START + pd.Timedelta(days=20)          # first 21 days = what the rule may see
obs = df[df["report_date"] <= CUTOFF].copy()
later = df[df["report_date"] > CUTOFF].copy()   # NEVER used for the score
print("Observation window:", obs["report_date"].min().date(), "to", obs["report_date"].max().date(), "| rows:", len(obs))
print("Later window (check only):", later["report_date"].min().date(), "to", later["report_date"].max().date(), "| rows:", len(later))

def agg_pages(d):
    g = d.groupby(KEY).agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        sum_pos=("gsc_sum_position", "sum"),
        days_active=("active", "sum"),
        max_day_impr=("gsc_impressions", "max"),
    ).reset_index()
    return g

pages = agg_pages(obs)
pages = pages[pages["impressions"] > 0].copy()
pages["ctr"] = pages["clicks"] / pages["impressions"]

# GSC sum_position is usually zero-indexed (position 1 -> 0). Heuristic: if any raw average is below 1, add 1.
raw_pos = pages["sum_pos"] / pages["impressions"]
POSITION_OFFSET = 1 if raw_pos.min() < 1 else 0
pages["avg_position"] = raw_pos + POSITION_OFFSET
print("Position offset used:", POSITION_OFFSET, "(heuristic - check the range below)")
print(pages["avg_position"].describe().round(2).to_string())
print("Pages with impressions in window:", len(pages))

Observation window: 2026-03-01 to 2026-03-21 | rows: 2366418
Later window (check only): 2026-03-22 to 2026-03-31 | rows: 1244643
Position offset used: 1 (heuristic - check the range below)
count    162301.00
mean         16.97
std          18.01
min           1.00
25%           5.89
50%           9.22
75%          21.33
max         310.00
Pages with impressions in window: 162301


In [5]:
# ---- Verdict helper (same logic for both signals) ----
def verdict(table, col, direction, flat_tol=0.10):
    # direction = +1 if we expect the metric to rise bucket by bucket, -1 if we expect it to fall
    vals = table[col].to_numpy(float)
    rel_range = (vals.max() - vals.min()) / max(abs(vals).max(), 1e-9)
    rho = pd.Series(range(len(vals))).corr(pd.Series(vals), method="spearman")
    if rel_range < flat_tol:
        v = "FALSE"        # basically flat: the signal does nothing
    elif rho * direction >= 0.9:
        v = "CONFIRMED"
    elif rho * direction <= -0.9:
        v = "OPPOSITE"
    else:
        v = "MIXED"
    return v, rho, rel_range

# ---- SIGNAL 1 (flag-linked, CTR-fix logic): does CTR fall as position gets worse? ----
POS_BINS = [0, 3, 5, 10, 20, 1000]
POS_LABELS = ["1-3", "3-5", "5-10", "10-20", "20+"]
s1 = pages[pages["impressions"] >= 100].copy()
s1["pos_bucket"] = pd.cut(s1["avg_position"], bins=POS_BINS, labels=POS_LABELS)
t1 = s1.groupby("pos_bucket", observed=True).agg(
    n_pages=("ctr", "size"), impressions=("impressions", "sum"), clicks=("clicks", "sum")).reset_index()
t1["pooled_ctr"] = t1["clicks"] / t1["impressions"]
print("SIGNAL 1 - CTR by average position bucket (pages with >=100 impressions)")
print(t1.round(4).to_string(index=False))
v1, rho1, rng1 = verdict(t1, "pooled_ctr", direction=-1)
print(f"\nSIGNAL 1 VERDICT: {v1}  (spearman across buckets = {rho1:.2f}, relative range = {rng1:.0%}, n pages = {len(s1)})")
print("Small buckets (n<30) to treat carefully:", t1.loc[t1["n_pages"] < 30, "pos_bucket"].tolist())

SIGNAL 1 - CTR by average position bucket (pages with >=100 impressions)
pos_bucket  n_pages  impressions  clicks  pooled_ctr
       1-3     4086      8233598   29260      0.0036
       3-5    14367     44284283  186788      0.0042
      5-10    31991     62380576  192165      0.0031
     10-20    18424     21086243   69184      0.0033
       20+    20044     41297461   59724      0.0014

SIGNAL 1 VERDICT: MIXED  (spearman across buckets = -0.80, relative range = 66%, n pages = 88912)
Small buckets (n<30) to treat carefully: []


In [6]:
# ---- SIGNAL 2 (flag-linked, volume / quick-win logic): do higher-volume keywords get more impressions? ----
cont = df_content[["content_hash_id", "search_volume"]].drop_duplicates("content_hash_id")
s2 = pages.merge(cont, on="content_hash_id", how="left")
n_before = len(s2)
s2 = s2[s2["search_volume"].notna() & (s2["search_volume"] > 0)].copy()
print(f"Pages with a usable search_volume: {len(s2)} of {n_before} (dropped {n_before - len(s2)} with missing/zero volume)")

s2["vol_bucket"] = pd.qcut(s2["search_volume"], q=4, duplicates="drop")
t2 = s2.groupby("vol_bucket", observed=True).agg(
    n_pages=("impressions", "size"), mean_impressions=("impressions", "mean"),
    median_impressions=("impressions", "median")).reset_index()
t2["vol_bucket"] = t2["vol_bucket"].astype(str)
print("\nSIGNAL 2 - impressions per page by search-volume bucket (quartiles, low -> high)")
print(t2.round(1).to_string(index=False))
v2, rho2, rng2 = verdict(t2, "mean_impressions", direction=+1)
print(f"\nSIGNAL 2 VERDICT: {v2}  (spearman across buckets = {rho2:.2f}, relative range = {rng2:.0%}, n pages = {len(s2)})")

Pages with a usable search_volume: 90760 of 162301 (dropped 71541 with missing/zero volume)

SIGNAL 2 - impressions per page by search-volume bucket (quartiles, low -> high)
      vol_bucket  n_pages  mean_impressions  median_impressions
   (9.999, 20.0]    52575            1098.8               196.0
    (20.0, 50.0]    15649            1234.4               192.0
(50.0, 368000.0]    22536            1173.6               167.0

SIGNAL 2 VERDICT: MIXED  (spearman across buckets = 0.50, relative range = 11%, n pages = 90760)


### How I read the two verdicts
- The verdicts are **computed by the code above** from the printed tables, not typed by hand. Read the printed `VERDICT` lines before trusting the rule.
- **If signal 1 is CONFIRMED:** the rule's idea (compare each page with what is normal for its position) is sound. **If it is OPPOSITE, MIXED or FALSE:** the rule is built on sand; say so in your final notes, a clearly explained negative is a win.
- **Signal 2** only affects how much I trust impressions as the size of the prize. The rule itself does not use search volume, so a weak signal 2 does not break the rule.
- Both tables are bucket averages, so they show direction, not cause.

## 2. Build the ranked queue (writes the CSV)

*Score = estimated missed clicks. One reason code. One action label.*

Expected CTR for each position is the **pooled CTR of pages in the same position band** (observation window only). This is in-sample, so it is a norm for "typical at this position", not a forecast.

In [7]:
# ---- The rule ----
MIN_IMPR, MAX_POS, MIN_POS = 100, 20, 1
RULE_BINS = [0, 1.5, 2.5, 3.5, 5, 7, 10, 15, 20]
RULE_LABELS = ["<=1.5", "1.5-2.5", "2.5-3.5", "3.5-5", "5-7", "7-10", "10-15", "15-20"]

pages["eligible"] = (pages["impressions"] >= MIN_IMPR) & (pages["avg_position"] >= MIN_POS) & (pages["avg_position"] <= MAX_POS)
pages["rule_bin"] = pd.cut(pages["avg_position"], bins=RULE_BINS, labels=RULE_LABELS)

norm = (pages[pages["eligible"]].groupby("rule_bin", observed=True)
        .agg(n_pages=("ctr", "size"), impressions=("impressions", "sum"), clicks=("clicks", "sum")).reset_index())
norm["expected_ctr"] = norm["clicks"] / norm["impressions"]
print("Expected CTR by position band (eligible pages, observation window):")
print(norm.round(4).to_string(index=False))

pages = pages.merge(norm[["rule_bin", "expected_ctr"]], on="rule_bin", how="left")
gap = (pages["expected_ctr"] - pages["ctr"]).clip(lower=0)
pages["score"] = np.where(pages["eligible"], gap * pages["impressions"], 0.0)   # estimated missed clicks
pages["reason_code"] = np.where(pages["score"] > 0, "CTR_BELOW_POSITION_NORM", "NO_SIGNAL")
pages["action"] = np.where(pages["score"] > 0, "REWRITE_TITLE_AND_META", "NO_ACTION")
pages["spike_share"] = pages["max_day_impr"] / pages["impressions"]

queue = pages.sort_values(["score", "impressions"], ascending=[False, False]).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue_cols = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
              "impressions", "clicks", "ctr", "avg_position", "expected_ctr", "days_active", "spike_share"]

os.makedirs("work/outputs", exist_ok=True)
queue[queue_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

n_flagged = int((queue["score"] > 0).sum())
print(f"\nQueue written: work/outputs/baseline_action_score.csv | rows: {len(queue)} | flagged: {n_flagged} | eligible: {int(queue['eligible'].sum())}")
print("Score = estimated missed clicks; total over flagged pages:", round(queue["score"].sum(), 1))
queue[queue_cols].head(10).round(4)

Expected CTR by position band (eligible pages, observation window):
rule_bin  n_pages  impressions  clicks  expected_ctr
   <=1.5      199       300962     173        0.0006
 1.5-2.5     1835      3368819    9182        0.0027
 2.5-3.5     4982     12961282   59003        0.0046
   3.5-5    11437     35886818  147690        0.0041
     5-7    15532     37627935  118911        0.0032
    7-10    16459     24752641   73254        0.0030
   10-15    11541     12381246   41874        0.0034
   15-20     6883      8704997   27310        0.0031

Queue written: work/outputs/baseline_action_score.csv | rows: 162301 | flagged: 45559 | eligible: 68868
Score = estimated missed clicks; total over flagged pages: 168717.1


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,ctr,avg_position,expected_ctr,days_active,spike_share
0,1,client_62f4a7e64f5e0096,content_34a70fea29d15f24,412.2982,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,106015,24,0.0002,4.1476,0.0041,21,0.3679
1,2,client_20259bd6705d81d4,content_82e35c4845e6c391,266.1341,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,96942,38,0.0004,19.3437,0.0031,21,0.0927
2,3,client_62f4a7e64f5e0096,content_f6116743b00afc2d,264.3621,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,81123,10,0.0001,10.8577,0.0034,21,0.1072
3,4,client_62f4a7e64f5e0096,content_acbcc847f8996314,262.1426,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,109622,189,0.0017,4.4552,0.0041,21,0.0791
4,5,client_62f4a7e64f5e0096,content_7c6373141eae744a,259.8711,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,100270,57,0.0006,6.8131,0.0032,21,0.1261
5,6,client_e547b89c05043229,content_306bc78dff1eb683,199.7704,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,49156,24,0.0005,2.6257,0.0046,19,0.0780
6,7,client_73cda7b4e4f265ea,content_8e1334d6356668e3,199.6271,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,63486,1,0.0000,5.8476,0.0032,21,0.1468
7,8,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,167.8477,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,60075,22,0.0004,5.6961,0.0032,21,0.4071
8,9,client_62f4a7e64f5e0096,content_36fc1ee501ec072d,165.9167,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,56616,13,0.0002,6.5950,0.0032,21,0.0827
9,10,client_62f4a7e64f5e0096,content_945d6ff91386c817,162.1842,CTR_BELOW_POSITION_NORM,REWRITE_TITLE_AND_META,55816,3,0.0001,9.3618,0.0030,21,0.6695


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.* The first 10 are the required review; rows 11-20 are the optional extension.

In [8]:
def confidence(r):
    if r.impressions >= 1000 and r.days_active >= 14 and r.spike_share < 0.30:
        return "HIGH (lots of impressions, steady over many days)"
    if r.impressions >= 300 and r.days_active >= 7:
        return "MEDIUM (decent volume, shorter history)"
    return "LOW (thin evidence)"

def wrong_if(r):
    reasons = []
    if r.spike_share >= 0.30:
        reasons.append(f"one day drove {r.spike_share:.0%} of impressions, so a one-off spike could fake the gap")
    if r.days_active < 10:
        reasons.append(f"only {int(r.days_active)} active days, too thin to trust")
    if r.avg_position > 10:
        reasons.append("page-2 average position: low CTR may be normal there")
    reasons.append("the query is branded/navigational or a SERP feature (AI box, snippet) takes the clicks, so a rewrite would not help")
    reasons.append("average position hides big day-to-day swings")
    return "; or ".join(reasons[:2])

top20 = queue[queue["score"] > 0].head(20).copy()
top20["confidence"] = top20.apply(confidence, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)
top20["why_it_is_here"] = top20.apply(
    lambda r: f"CTR {r.ctr:.1%} vs {r.expected_ctr:.1%} typical at position ~{r.avg_position:.1f}, {int(r.impressions):,} impressions (~{r.score:.0f} clicks missed)", axis=1)

print(f"TOP-10 REVIEW (required) | action = REWRITE_TITLE_AND_META | reason code = CTR_BELOW_POSITION_NORM\n")
for r in top20.head(10).itertuples():
    print(f"#{r.rank}  [{str(r.content_hash_id)[:8]}]  {r.why_it_is_here}")
    print(f"     confidence: {r.confidence}")
    print(f"     wrong if:   {r.what_would_make_it_wrong}\n")

top20[["rank", "content_hash_id", "action", "reason_code", "score", "confidence", "why_it_is_here", "what_would_make_it_wrong"]].round(2)

TOP-10 REVIEW (required) | action = REWRITE_TITLE_AND_META | reason code = CTR_BELOW_POSITION_NORM

#1  [content_]  CTR 0.0% vs 0.4% typical at position ~4.1, 106,015 impressions (~412 clicks missed)
     confidence: MEDIUM (decent volume, shorter history)
     wrong if:   one day drove 37% of impressions, so a one-off spike could fake the gap; or the query is branded/navigational or a SERP feature (AI box, snippet) takes the clicks, so a rewrite would not help

#2  [content_]  CTR 0.0% vs 0.3% typical at position ~19.3, 96,942 impressions (~266 clicks missed)
     confidence: HIGH (lots of impressions, steady over many days)
     wrong if:   page-2 average position: low CTR may be normal there; or the query is branded/navigational or a SERP feature (AI box, snippet) takes the clicks, so a rewrite would not help

#3  [content_]  CTR 0.0% vs 0.3% typical at position ~10.9, 81,123 impressions (~264 clicks missed)
     confidence: HIGH (lots of impressions, steady over many days)
     wro

,rank,content_hash_id,action,reason_code,score,confidence,why_it_is_here,what_would_make_it_wrong
0,1,content_34a70fea29d15f24,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,412.30,"MEDIUM (decent volume, shorter history)","CTR 0.0% vs 0.4% typical at position ~4.1, 106...","one day drove 37% of impressions, so a one-off..."
1,2,content_82e35c4845e6c391,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,266.13,"HIGH (lots of impressions, steady over many days)","CTR 0.0% vs 0.3% typical at position ~19.3, 96...",page-2 average position: low CTR may be normal...
2,3,content_f6116743b00afc2d,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,264.36,"HIGH (lots of impressions, steady over many days)","CTR 0.0% vs 0.3% typical at position ~10.9, 81...",page-2 average position: low CTR may be normal...
3,4,content_acbcc847f8996314,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,262.14,"HIGH (lots of impressions, steady over many days)","CTR 0.2% vs 0.4% typical at position ~4.5, 109...",the query is branded/navigational or a SERP fe...
4,5,content_7c6373141eae744a,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,259.87,"HIGH (lots of impressions, steady over many days)","CTR 0.1% vs 0.3% typical at position ~6.8, 100...",the query is branded/navigational or a SERP fe...
5,6,content_306bc78dff1eb683,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,199.77,"HIGH (lots of impressions, steady over many days)","CTR 0.0% vs 0.5% typical at position ~2.6, 49,...",the query is branded/navigational or a SERP fe...
6,7,content_8e1334d6356668e3,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,199.63,"HIGH (lots of impressions, steady over many days)","CTR 0.0% vs 0.3% typical at position ~5.8, 63,...",the query is branded/navigational or a SERP fe...
7,8,content_1642f339bd6e7c8d,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,167.85,"MEDIUM (decent volume, shorter history)","CTR 0.0% vs 0.3% typical at position ~5.7, 60,...","one day drove 41% of impressions, so a one-off..."
8,9,content_36fc1ee501ec072d,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,165.92,"HIGH (lots of impressions, steady over many days)","CTR 0.0% vs 0.3% typical at position ~6.6, 56,...",the query is branded/navigational or a SERP fe...
9,10,content_945d6ff91386c817,REWRITE_TITLE_AND_META,CTR_BELOW_POSITION_NORM,162.18,"MEDIUM (decent volume, shorter history)","CTR 0.0% vs 0.3% typical at position ~9.4, 55,...","one day drove 67% of impressions, so a one-off..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Where the rule is weak (structural, not just in this run):**
- **Branded / navigational queries:** people search a name and click the first result, or read the answer on the page. Low CTR can be normal.
- **SERP features:** AI answers, snippets, and maps take clicks that a title rewrite cannot win back.
- **Average position hides swings:** a page that is #1 on some days and #15 on others looks like position 8.
- **In-sample norm:** the expected CTR comes from the same month, so it describes "typical", not "achievable".
- **Page level, not query level:** one page can rank for many queries with very different CTRs.

In [9]:
# ---- Weak picks inside my own top 20 ----
w = top20.copy()
w["weak_thin"] = w["impressions"] < 300
w["weak_spike"] = w["spike_share"] >= 0.30
w["weak_few_days"] = w["days_active"] < 10
w["weak_page2"] = w["avg_position"] > 10
w["n_weak_flags"] = w[["weak_thin", "weak_spike", "weak_few_days", "weak_page2"]].sum(axis=1)
print("Top-20 picks with at least one weakness flag:", int((w["n_weak_flags"] > 0).sum()), "of", len(w))
print(w.loc[w["n_weak_flags"] > 0, ["rank", "impressions", "avg_position", "days_active", "spike_share", "n_weak_flags"]].round(2).to_string(index=False))

# ---- Directional persistence check (uses the LATER window, only for checking, never for the score) ----
later_pages = agg_pages(later)
later_pages = later_pages[later_pages["impressions"] >= 50].copy()
later_pages["ctr_later"] = later_pages["clicks"] / later_pages["impressions"]
chk = queue[queue["score"] > 0][KEY + ["rank", "expected_ctr"]].merge(later_pages[KEY + ["ctr_later", "impressions"]], on=KEY, how="inner")

def persist(d, name):
    if len(d) == 0:
        print(f"{name}: n = 0 (no rows with >=50 later impressions)"); return {"n": 0}
    share = float((d["ctr_later"] < d["expected_ctr"]).mean())
    out = {"n": int(len(d)), "share_still_below_expected": round(share, 3),
           "median_ctr_later": round(float(d["ctr_later"].median()), 4),
           "median_expected_ctr": round(float(d["expected_ctr"].median()), 4)}
    print(f"{name}: {out}")
    return out

print("\nPersistence check (directional; expect some regression to the mean):")
p20 = persist(chk[chk["rank"] <= 20], "Top 20 flagged")
p100 = persist(chk[chk["rank"] <= 100], "Top 100 flagged")
pall = persist(chk, "All flagged")

Top-20 picks with at least one weakness flag: 7 of 20
 rank  impressions  avg_position  days_active  spike_share  n_weak_flags
    1       106015          4.15           21         0.37             1
    2        96942         19.34           21         0.09             1
    3        81123         10.86           21         0.11             1
    8        60075          5.70           21         0.41             1
   10        55816          9.36           21         0.67             1
   15        50543          3.12           21         0.44             1
   17        89534         18.83           21         0.09             1

Persistence check (directional; expect some regression to the mean):
Top 20 flagged: {'n': 20, 'share_still_below_expected': 1.0, 'median_ctr_later': 0.0008, 'median_expected_ctr': 0.0033}
Top 100 flagged: {'n': 99, 'share_still_below_expected': 1.0, 'median_ctr_later': 0.0008, 'median_expected_ctr': 0.0034}
All flagged: {'n': 41624, 'share_still_below_expect

In [10]:
# ---- Leakage check ----
USED_COLUMNS = ["client_hash_id", "content_hash_id", "report_date", "gsc_impressions", "gsc_clicks", "gsc_sum_position"]
banned_fragments = ["sessions_paid", "paid", "ai_", "ga4_", "label", "has_organic_engagement", "product", "flag"]
bad = [c for c in USED_COLUMNS if any(b in c for b in banned_fragments)]
assert not bad, f"Banned inputs found: {bad}"
assert obs["report_date"].max() <= CUTOFF, "Score used dates after the cutoff"
assert later["report_date"].min() > obs["report_date"].max(), "Windows overlap"
print("Leakage check PASSED:")
print(" - Inputs used:", USED_COLUMNS)
print(" - No paid, AI, GA4, product-flag, or label columns used.")
print(f" - Score window ends {obs['report_date'].max().date()}; later window starts {later['report_date'].min().date()} and is used ONLY for the persistence check.")
print(" - gsc_clicks is used as part of the rule's CTR input from the observation window; no later or label-derived value feeds the score.")

# ---- Receipt (OK to commit: no data rows, only counts and verdicts) ----
metrics = {
    "assignment": "ML-07 baseline action score",
    "window_observation": [str(obs["report_date"].min().date()), str(obs["report_date"].max().date())],
    "window_check_only": [str(later["report_date"].min().date()), str(later["report_date"].max().date())],
    "position_offset": int(POSITION_OFFSET),
    "signal_1_ctr_vs_position": {"verdict": v1, "spearman": round(float(rho1), 3), "n_pages": int(len(s1))},
    "signal_2_volume_vs_impressions": {"verdict": v2, "spearman": round(float(rho2), 3), "n_pages": int(len(s2))},
    "queue_rows": int(len(queue)), "flagged_rows": n_flagged,
    "persistence_top20": p20, "persistence_top100": p100, "persistence_all_flagged": pall,
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nWrote work/outputs/baseline_metrics.json")

Leakage check PASSED:
 - Inputs used: ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position']
 - No paid, AI, GA4, product-flag, or label columns used.
 - Score window ends 2026-03-21; later window starts 2026-03-22 and is used ONLY for the persistence check.
 - gsc_clicks is used as part of the rule's CTR input from the observation window; no later or label-derived value feeds the score.

Wrote work/outputs/baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, tokens, or private queries anywhere (only hashed ids appear, and the CSV stays out of git)
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Both signal verdicts printed with n; at least one is flag-linked (both are)
- [ ] Committed to my repo under `work/notebooks/` (plus `work/outputs/baseline_metrics.json`), then submit the repo URL on the card. Done.